# Compare Downloaded W&B Runs

Use this notebook to compare metric histories across multiple downloaded W&B runs stored under `wandb_downloads/*`.

## What this notebook does

1. Takes an explicit list of W&B run-directory paths.
2. Loads each run's `run.json`, `summary.json`, and `history.jsonl`.
3. Builds a short summary table for the loaded runs.
4. Plots one stacked figure with one subplot per requested metric.
5. Draws a solid raw line and a dotted 5-point running average for each run.

The notebook is read-only with respect to the downloaded artifacts. It only visualizes the exported W&B data already present in the repo.

In [ ]:
from __future__ import annotations

import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "sumo_rl").exists() and (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not locate the repo root from the current working directory.")


ROOT = find_repo_root()
DEFAULT_WANDB_ROOT = ROOT / "wandb_downloads"

plt.rcParams["figure.dpi"] = 130
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

print(f"Repo root: {ROOT}")
print(f"Default W&B download root: {DEFAULT_WANDB_ROOT}")

In [ ]:
RUN_PATHS: list[str] = [
    # Example:
    str(DEFAULT_WANDB_ROOT / "jv-fuisl-vietnamese-german-university" / "marl-traffic-gat" / "baseline" / "1c8l6ojg__resco_cologne8__static_max_pressure"),
    str(DEFAULT_WANDB_ROOT / "jv-fuisl-vietnamese-german-university" / "marl-traffic-gat" / "dcrnn" / "0enisfv9__ingolstadt7__ppo_dcrnn_mlp__013807"),
]

METRICS: list[str] = [
    # Example:
    "train/resco_delay_mean",
    "validation/resco_delay_mean",
]

SMOOTHING_WINDOW = 5
RUN_LABELS: dict[str, str] | None = None
FIG_WIDTH = 14
ROW_HEIGHT = 4
RAW_LINE_ALPHA = 0.35
RAW_LINE_WIDTH = 1.5
SMOOTH_LINE_WIDTH = 2.3
LEGEND_NCOL = 2

In [ ]:
REQUIRED_EXPORT_FILES = ("run.json", "summary.json", "history.jsonl")


def resolve_run_path(run_path: str | Path) -> Path:
    path = Path(run_path).expanduser()
    if not path.is_absolute():
        path = (ROOT / path).resolve()
    else:
        path = path.resolve()

    if not path.exists() or not path.is_dir():
        raise FileNotFoundError(f"Run directory does not exist: {path}")

    missing = [name for name in REQUIRED_EXPORT_FILES if not (path / name).exists()]
    if missing:
        missing_text = ", ".join(missing)
        raise FileNotFoundError(f"Run directory is missing required files ({missing_text}): {path}")
    return path


def read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))


def read_history_jsonl(path: Path) -> list[dict]:
    rows: list[dict] = []
    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f"Invalid JSON on line {line_number} of {path}") from exc
    return rows


def derive_run_label(run_dir: Path, run_meta: dict, run_labels: dict[str, str] | None = None) -> str:
    if run_labels:
        for key in (str(run_dir), run_dir.as_posix(), run_dir.name):
            if key in run_labels and str(run_labels[key]).strip():
                return str(run_labels[key]).strip()
    name = str(run_meta.get("name") or "").strip()
    return name or run_dir.name


def choose_x_axis(metric: str, frame: pd.DataFrame) -> str:
    preferred = "_step"
    if metric.startswith("train/"):
        preferred = "train/env_step"
    elif metric.startswith("validation/"):
        preferred = "validation/env_step"

    candidates = [preferred, "_step"]
    for candidate in candidates:
        if candidate in frame.columns and frame[candidate].notna().any():
            return candidate
    raise KeyError(f"Could not find a usable x-axis column for metric '{metric}'.")


def prepare_metric_series(frame: pd.DataFrame, metric: str, smoothing_window: int) -> tuple[pd.DataFrame, str] | None:
    if metric not in frame.columns:
        return None

    x_column = choose_x_axis(metric, frame)
    metric_frame = frame[[x_column, metric]].copy()
    metric_frame[x_column] = pd.to_numeric(metric_frame[x_column], errors="coerce")
    metric_frame[metric] = pd.to_numeric(metric_frame[metric], errors="coerce")
    metric_frame = metric_frame.dropna(subset=[x_column, metric])
    if metric_frame.empty:
        return None

    metric_frame = metric_frame.sort_values(x_column, kind="stable").reset_index(drop=True)
    metric_frame["running_average"] = metric_frame[metric].rolling(window=smoothing_window, min_periods=1).mean()
    return metric_frame, x_column


def load_run_bundle(run_path: str | Path, run_labels: dict[str, str] | None = None) -> dict:
    run_dir = resolve_run_path(run_path)
    run_meta = read_json(run_dir / "run.json")
    summary = read_json(run_dir / "summary.json")
    history_rows = read_history_jsonl(run_dir / "history.jsonl")
    history_frame = pd.DataFrame(history_rows)

    return {
        "run_dir": run_dir,
        "run_meta": run_meta,
        "summary": summary,
        "history": history_frame,
        "label": derive_run_label(run_dir, run_meta, run_labels=run_labels),
    }


def summarize_loaded_runs(run_bundles: list[dict]) -> pd.DataFrame:
    rows = []
    for bundle in run_bundles:
        frame = bundle["history"]
        rows.append(
            {
                "label": bundle["label"],
                "algorithm": bundle["run_meta"].get("job_type"),
                "tags": ", ".join(bundle["run_meta"].get("tags") or []),
                "history_rows": len(frame),
                "history_columns": len(frame.columns),
                "path": str(bundle["run_dir"]),
            }
        )
    return pd.DataFrame(rows)


def build_color_map(labels: list[str]) -> dict[str, tuple[float, float, float, float]]:
    palette_name = "tab10" if len(labels) <= 10 else "tab20"
    cmap = plt.get_cmap(palette_name)
    if hasattr(cmap, "colors"):
        colors = list(cmap.colors)
    else:
        colors = [cmap(index / max(len(labels), 1)) for index in range(len(labels))]
    return {label: colors[index % len(colors)] for index, label in enumerate(labels)}


def plot_metric_comparison(
    run_bundles: list[dict],
    metrics: list[str],
    *,
    smoothing_window: int = 5,
    fig_width: int = 14,
    row_height: int = 4,
):
    if not run_bundles:
        raise ValueError("No runs were loaded. Populate RUN_PATHS first.")
    if not metrics:
        raise ValueError("No metrics were requested. Populate METRICS first.")
    if smoothing_window < 1:
        raise ValueError("SMOOTHING_WINDOW must be at least 1.")

    labels = [bundle["label"] for bundle in run_bundles]
    color_map = build_color_map(labels)
    figure, axes = plt.subplots(
        nrows=len(metrics),
        ncols=1,
        figsize=(fig_width, row_height * len(metrics)),
        squeeze=False,
        sharex=False,
    )
    axes = axes.flatten()
    legend_handles: dict[str, object] = {}

    for axis, metric in zip(axes, metrics):
        plotted_count = 0
        x_axis_label = None

        for bundle in run_bundles:
            label = bundle["label"]
            frame = bundle["history"]
            color = color_map[label]

            try:
                prepared = prepare_metric_series(frame, metric, smoothing_window)
            except KeyError:
                warnings.warn(f"Skipping metric '{metric}' for run '{label}' because no usable x-axis was found.")
                continue

            if prepared is None:
                warnings.warn(f"Skipping metric '{metric}' for run '{label}' because the series is missing or non-numeric.")
                continue

            metric_frame, x_column = prepared
            x_axis_label = x_axis_label or x_column

            axis.plot(
                metric_frame[x_column],
                metric_frame[metric],
                color=color,
                linewidth=RAW_LINE_WIDTH,
                alpha=RAW_LINE_ALPHA,
                linestyle="-",
            )
            (handle,) = axis.plot(
                metric_frame[x_column],
                metric_frame["running_average"],
                color=color,
                linewidth=SMOOTH_LINE_WIDTH,
                linestyle=":",
                label=label,
            )
            legend_handles.setdefault(label, handle)
            plotted_count += 1

        axis.set_title(metric)
        axis.set_ylabel(metric)
        axis.set_xlabel(x_axis_label or "step")
        axis.grid(True, alpha=0.25)

        if plotted_count == 0:
            axis.text(
                0.5,
                0.5,
                "No plottable data found for this metric across the selected runs.",
                ha="center",
                va="center",
                transform=axis.transAxes,
            )

    if legend_handles:
        figure.legend(
            legend_handles.values(),
            legend_handles.keys(),
            loc="upper center",
            ncol=min(LEGEND_NCOL, max(1, len(legend_handles))),
            frameon=False,
            bbox_to_anchor=(0.5, 1.02),
        )

    figure.suptitle(f"W&B metric comparison across {len(run_bundles)} run(s)", y=1.04)
    figure.tight_layout()
    plt.show()


def collect_available_metrics(run_bundles: list[dict], prefix: str = "") -> list[str]:
    metric_names: set[str] = set()
    skip_prefixes = ("_", "debug/", "validation/actions_", "validation/phase_queue/")
    for bundle in run_bundles:
        for column in bundle["history"].columns:
            if any(column.startswith(skip_prefix) for skip_prefix in skip_prefixes):
                continue
            if prefix and not column.startswith(prefix):
                continue
            metric_names.add(str(column))
    return sorted(metric_names)


In [ ]:
loaded_runs = [load_run_bundle(run_path, run_labels=RUN_LABELS) for run_path in RUN_PATHS]
run_summary = summarize_loaded_runs(loaded_runs)

display(run_summary)

if loaded_runs:
    print("Example available train metrics:")
    print(collect_available_metrics(loaded_runs, prefix="train/")[:25])
    print()
    print("Example available validation metrics:")
    print(collect_available_metrics(loaded_runs, prefix="validation/")[:25])

In [ ]:
plot_metric_comparison(
    loaded_runs,
    METRICS,
    smoothing_window=SMOOTHING_WINDOW,
    fig_width=FIG_WIDTH,
    row_height=ROW_HEIGHT,
)